In [2]:
from appworld import AppWorld, load_task_ids
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import List, Literal, Optional, Tuple
from openai import OpenAI

In [3]:
client = OpenAI(
    base_url="https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1",
    api_key="dummy",  # or your real key if you used --api-key
)

In [4]:
resp = client.chat.completions.create(
    model="microsoft/Phi-3-mini-4k-instruct",
    messages=[
        {"role": "user", "content": "Explain gravity in one sentence."}
    ],
    max_tokens=50,
    logprobs=True,          # <-- REQUIRED
)

In [8]:
from baseline.templates import Template
from baseline.config import Config

# models.py
from pydantic import BaseModel
from typing import List, Literal


class Message(BaseModel):
    """Single conversation message"""
    role: Literal["system", "user", "assistant"]
    content: str
    log_probs: Optional[List[Tuple[str, float]]] = None


class AgentState(BaseModel):
    conversation_history: List[Message] = Field(default_factory=list)
    log_probs: List[float] = Field(default_factory=list)
    iteration: int = 0
    done: bool = False
    max_iters: int = 50
    
    @property 
    def should_continue(self):
        return self.iteration < self.max_iters and not self.done
    
    def total_chars(self):
        return sum(len(msg.content) for msg in self.conversation_history)

In [9]:
import re

def message_parser_with_position(message: str) -> Tuple[Optional[str], Optional[int], Optional[int]]:
    """
    Extract code from markdown code blocks and return code + positions.
    Returns (code, start_pos, end_pos) where positions mark the full code block including ```.
    """
    pattern = r'```(?:python)?\n(.*?)```'
    match = re.search(pattern, message, re.DOTALL)
    
    if match:
        code = match.group(1).strip()
        # match.start() is the position of the opening ```
        # match.end() is the position after the closing ```
        return code, match.start(), match.end()
    
    # Fallback for non-markdown code
    if message.strip().startswith(('print(', 'apis.', 'import ', 'from ')):
        return message.strip(), 0, len(message)
    
    return None, None, None

def truncate_message_history(
    conversation_history: List[Message], 
    threshold: int
) -> List[Message]:
    """
    Truncate conversation history if it exceeds the character threshold.
    Keeps the system message and most recent messages.
    """
    total_chars = sum(len(msg.content) for msg in conversation_history)
    
    if total_chars <= threshold:
        return conversation_history
    
    # Always keep the first message (initial prompt with instructions)
    truncated = [conversation_history[0]]
    
    # Keep most recent messages until we're under threshold
    recent_messages = []
    current_chars = len(conversation_history[0].content)
    
    # Work backwards from most recent
    for msg in reversed(conversation_history[1:]):
        msg_chars = len(msg.content)
        if current_chars + msg_chars <= threshold:
            recent_messages.insert(0, msg)
            current_chars += msg_chars
        else:
            break
    
    truncated.extend(recent_messages)
    return truncated

In [10]:
from dataclasses import dataclass
import os
from dotenv import load_dotenv

load_dotenv()

@dataclass 
class Config:
    # Agent parameters
    max_iters: int = 50  # Match the paper's baseline
    
    # OpenAI parameters
    openai_api_key: str = os.getenv("OPENAI_API_KEY")
    service: str = "vLLM" # can set to OpenAI or TogetherAI
    togetherai_api_key: str = os.getenv("TOGETHER_AI")
    base_model: str = os.getenv("VLLM_MODEL") 
    max_tokens: int = 512
    temperature: float = 0.0
    
    # Context management
    truncation_threshold: int = 12000  # Characters, not tokens

    # AppWorld Root
    os.environ["APPWORLD_ROOT"] = os.getenv("APPWORLD_ROOT")
    
    @classmethod
    def for_model(cls, model_name: str):
        """Factory method for different model configs"""
        configs = {
            "gpt-4o": cls(base_model="gpt-4o-2024-05-13"),
            "gpt-4": cls(base_model="gpt-4-turbo-2024-04-09"),
            "o1": cls(
                base_model="o1-preview-2024-09-12",
                temperature=1.0,  # o1 requires temperature=1
                max_tokens=4000
            )
        }
        return configs.get(model_name, cls())

config = Config()

In [11]:
max_iters: int = config.max_iters
max_tokens: int = config.max_tokens
temperature: float = config.temperature
base_model: str = config.base_model
truncation_threshold: int = config.truncation_threshold
template = Template()
state = AgentState(max_iters=config.max_iters)
seed = None

In [12]:
task_ids = load_task_ids("train") # loads train ids, other options: dev, test_normal, test_challenge
task_id = task_ids[0]
world = AppWorld(task_id=task_id)

In [13]:
first_name = "John"
last_name = "Smith"
email = "john@yahoo.com"
phone_number = "1234567894"

init_template = template.format_prompt(
    first_name, 
    last_name, 
    email, 
    phone_number, 
    world.task.instruction
)

In [14]:
K=6
sets=["train"]

In [15]:
train_ids = [
		    tid
		    for dataset_name in sets
		    for tid in load_task_ids(dataset_name)
		]

In [16]:
task_id = train_ids[0]

In [17]:
import uuid
random_uuid = uuid.uuid4()

In [18]:
task_result = {
					"task_id": task_id,
					"completed": False,
					"iterations": 0,
					"error": None,
					"result": None,
					"conversation_length": 0,
					"token_log_probs": None,
					"unit_tests": None,
					"overall_success": None,
					"uuid": random_uuid
				}

In [19]:
from typing import Union
class ReactAgent:
    def __init__(self, config: Config, return_log_probs: bool = False, seed: int = None) -> None:
        self.max_iters: int = config.max_iters  # Fixed: use instance
        self.max_tokens: int = 512
        self.temperature: float = config.temperature
        self.base_model: str = config.base_model
        self.truncation_threshold: int = config.truncation_threshold
        self.template = Template()
        self.state = AgentState(max_iters=config.max_iters)
        self.seed = seed
        if config.service == "OpenAI":
            self.client = OpenAI(api_key=config.openai_api_key)
        elif config.service == "TogetherAI":
            os.environ["TOGETHER_API_KEY"] = config.togetherai_api_key
            self.client = OpenAI(
                api_key=config.togetherai_api_key,
                base_url="https://api.together.xyz/v1"
            )
        elif config.service == "vLLM":
            openai_api_key = "EMPTY"
            openai_api_base = "https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1"
            self.client = OpenAI(
                api_key=openai_api_key,
                base_url=openai_api_base,
            )
        self.eval_tracker: Dict = {}  

        
    def initialize(
        self, 
        first_name: str, 
        last_name: str, 
        email: str, 
        phone_number: str, 
        task_instructions: str
    ) -> None:
        init_template = self.template.format_prompt(
            first_name, 
            last_name, 
            email, 
            phone_number, 
            task_instructions
        )
        self.state.conversation_history.append(
            Message(role="user", content=init_template)
        )
    
    def call_llm(self, return_log_probs: bool = True) -> Tuple[str, Union[None, List[Tuple]]]: 
        messages = truncate_message_history(
            self.state.conversation_history, 
            self.truncation_threshold
        )

        extra_args = {}
        if self.seed:
            extra_args["seed"] = self.seed
        if return_log_probs:
            extra_args["logprobs"] = True
            extra_args["top_logprobs"] = 1  # only need the generated token

        response = self.client.chat.completions.create(
            model=self.base_model,
            messages=[msg.dict() for msg in messages],
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            **extra_args,
        )

        choice = response.choices[0]
        text = choice.message.content

        if not return_log_probs:
            return text, None

        token_logprobs = []
        if choice.logprobs is not None:
            for item in choice.logprobs.content:
                token_logprobs.append((item.token, item.logprob))

        return text, token_logprobs
    
    def step(self, world):  
        llm_output, token_logprobs = self.call_llm(return_log_probs=True)
        
        # Log full output for analysis
        self.eval_tracker[f"iter_{self.state.iteration}_full_output"] = llm_output
        
        # Extract code and find its position
        code, code_start, code_end = message_parser_with_position(llm_output)
        observation_string = "No code block found in response."
        
        if code:
            try:
                observation = world.execute(code)
                observation_string = str(observation)
            except Exception as e:
                observation_string = f"Error: {str(e)}"
            
            # Truncate to include everything up to and including the code block
            truncated_response = llm_output[:code_end].strip()
            
            # Build up tokens until exact match
            truncated_logprobs = None
            if token_logprobs:
                reconstructed = ""
                truncated_logprobs = []
                
                for token, logprob in token_logprobs:
                    reconstructed += token
                    truncated_logprobs.append((token, logprob))
                    
                    if reconstructed.strip() == truncated_response:
                        break
                    # Safety check
                    if len(reconstructed) > len(truncated_response) + 10:
                        break
            
            self.state.conversation_history.append(
                Message(role="assistant", content=truncated_response, log_probs=truncated_logprobs)
            )
        else:
            # No code found - store full response
            self.state.conversation_history.append(
                Message(role="assistant", content=llm_output, log_probs=token_logprobs)
            )
        
        # Append real observation
        self.state.conversation_history.append(
            Message(role="user", content=f"Output:\n```\n{observation_string}\n```")
        )
        
        self.state.iteration += 1
        
        if world.task_completed():
            self.state.done = True
        
        return world
    
    def run(self, world):
        """
        Execute agent loop until completion or max iterations
        """
        while self.state.should_continue:
            world = self.step(world)
            
            if self.state.done:
                print(f"Task completed in {self.state.iteration} iterations")
                break
        
        if not self.state.done:
            print(f"Max iterations ({self.max_iters}) reached without completion")
        
        return world


In [20]:
agent = ReactAgent(config)

In [21]:
agent.initialize(
                first_name,
                last_name,
                email,
                phone_number,
                world.task.instruction
            )	

In [22]:
agent.run(world)

Max iterations (50) reached without completion


In [23]:
print(agent.state.conversation_history[0].content)

USER:
    I am your supervisor and you are a super intelligent AI Assistant whose job is to achieve my day-to-day tasks completely autonomously.

    To do this, you will need to interact with app/s (e.g., spotify, venmo etc) using their associated APIs on my behalf. 
    For this you will undertake a *multi-step conversation* using a python REPL environment. That is, you will write the 
    python code and the environment will execute it and show you the result, based on which, you will write python code 
    for the next step and so on, until you've achieved the goal. This environment will let you interact with app/s using their associated APIs on my behalf.

    Here are three key APIs that you need to know to get more information

    # To get a list of apps that are available to you.
    print(apis.api_docs.show_app_descriptions())

    # To get the list of apis under any app listed above, e.g. spotify
    print(apis.api_docs.show_api_descriptions(app_name='spotify'))

    # To ge

In [24]:
print(agent.state.conversation_history[1].content)

Okay. Lets first find which APIs are available to use in Spotify.
Code:
```python
print(apis.api_docs.show_api_descriptions(app_name='spotify'))
```


In [25]:
agent.state.conversation_history[1].log_probs

[('Okay', -0.7412142753601074),
 ('.', -0.3165111243724823),
 ('L', -0.9072583317756653),
 ('ets', -0.0003897384158335626),
 ('first', -0.3292463719844818),
 ('find', -0.06045417860150337),
 ('which', -0.16150923073291779),
 ('APIs', -0.044258084148168564),
 ('are', -0.0025912299752235413),
 ('available', -0.000567275274079293),
 ('to', -0.024503814056515694),
 ('use', -0.0010787388309836388),
 ('in', -0.10865531861782074),
 ('Sp', -0.043847426772117615),
 ('ot', -1.549708758830093e-05),
 ('ify', -1.1086402082582936e-05),
 ('.', -0.05449630692601204),
 ('\n', -0.03822099789977074),
 ('Code', -0.11245669424533844),
 (':', -0.0008985534077510238),
 ('\n', -0.0011038646334782243),
 ('```', -0.004141089040786028),
 ('python', -0.0005218812730163336),
 ('\n', -7.629103492945433e-05),
 ('print', -0.005716760642826557),
 ('(', -9.500529267825186e-05),
 ('apis', -3.397406908334233e-05),
 ('.', -1.8954096958623268e-05),
 ('api', -5.5549986427649856e-05),
 ('_', -2.264974000354414e-06),
 ('docs'

In [26]:
count = 0

for token, prob in agent.state.conversation_history[1].log_probs:
    count += len(token)

count

159

In [27]:
extra_args = {}

extra_args["logprobs"] = True
extra_args["top_logprobs"] = 1  

response = agent.client.chat.completions.create(
    model=agent.base_model,
    messages=[{"role": "user", "content": str(agent.state.conversation_history[0].content)}],
    temperature=agent.temperature,
    max_tokens=agent.max_tokens,
    **extra_args
)

In [28]:
choice = response.choices[0]
text = choice.message.content

token_logprobs = []
if choice.logprobs is not None:
    for item in choice.logprobs.content:
        token_logprobs.append((item.token, item.logprob))

In [29]:
# Extract code and find its position
llm_output = text

code, code_start, code_end = message_parser_with_position(llm_output)
observation_string = "No code block found in response."

In [30]:
code_end

149

In [31]:
# Truncate to include everything up to and including the code block
truncated_response = llm_output[:code_end].strip()

In [53]:
truncated_response

"Okay. Lets first find which APIs are available to use in Spotify.\nCode:\n```python\nprint(apis.api_docs.show_api_descriptions(app_name='spotify'))\n```"

In [55]:
# Build up tokens until exact match

truncated_logprobs = []
found_opening = False
backtick_count = 0

for i, (token, logprob) in enumerate(token_logprobs):
    truncated_logprobs.append((token, logprob))
    
    # Count backtick tokens
    if token == '```':
        backtick_count += 1
        if backtick_count == 1:
            found_opening = True
        elif backtick_count == 2 and found_opening:
            # Found closing backticks - stop here
            break


In [56]:
truncated_logprobs

[('Okay', -0.7412142753601074),
 ('.', -0.3165111243724823),
 ('L', -0.9072583317756653),
 ('ets', -0.0003897384158335626),
 ('first', -0.3292463719844818),
 ('find', -0.06045417860150337),
 ('which', -0.16150923073291779),
 ('APIs', -0.044258084148168564),
 ('are', -0.0025912299752235413),
 ('available', -0.000567275274079293),
 ('to', -0.024503814056515694),
 ('use', -0.0010787388309836388),
 ('in', -0.10865531861782074),
 ('Sp', -0.043847426772117615),
 ('ot', -1.549708758830093e-05),
 ('ify', -1.1086402082582936e-05),
 ('.', -0.05449630692601204),
 ('\n', -0.03822099789977074),
 ('Code', -0.11245669424533844),
 (':', -0.0008985534077510238),
 ('\n', -0.0011038646334782243),
 ('```', -0.004141089040786028),
 ('python', -0.0005218812730163336),
 ('\n', -7.629103492945433e-05),
 ('print', -0.005716760642826557),
 ('(', -9.500529267825186e-05),
 ('apis', -3.397406908334233e-05),
 ('.', -1.8954096958623268e-05),
 ('api', -5.5549986427649856e-05),
 ('_', -2.264974000354414e-06),
 ('docs'

In [43]:
for tup in truncated_logprobs:
    print(repr(tup[0]))

'Okay'
'.'
'L'
'ets'
'first'
'find'
'which'
'APIs'
'are'
'available'
'to'
'use'
'in'
'Sp'
'ot'
'ify'
'.'
'\n'
'Code'
':'
'\n'
'```'
'python'
'\n'
'print'
'('
'apis'
'.'
'api'
'_'
'docs'
'.'
'show'
'_'
'api'
'_'
'des'
'cri'
'ptions'
'('
'app'
'_'
'name'
"='"
'spot'
'ify'
"'))"
'\n'
'```'
'\n'
'\n'
'USER'
':'
'\n'
'Output'
